# Seed `silver_status_code_mapping`

**Owner:** Lead • **Run once** after Bronze ingestion completes (Phase 3).

Builds the canonical status mapping table that all Silver T2 transforms join against. Members **read** this table; only the lead writes it.

## How it works
1. Read all `*_status_codes` lookup tables from Bronze
2. Read distinct status values from each transactional table that carries a status column
3. Map each (source_system, raw_code) pair to a `canonical_status` from the glossary set: `OPEN`, `CONFIRMED`, `IN_FULFILMENT`, `DELIVERED`, `CANCELLED`, `RETURNED`, `REFUNDED`
4. Anything unmapped → `canonical_status = 'UNMAPPED'` + flag for lead review

Output: `NEXAMART_SILVER.silver_status_code_mapping` with columns:
- `source_system` (e.g. 'POS', 'EC', 'MARKETPLACE', 'NEXALOCAL')
- `raw_code` (the source system's status code)
- `raw_description`
- `canonical_status`
- `mapping_confidence` (1.00 deterministic from lookup, 0.80 inferred from transactional)
- 4 mandatory anomaly columns (rows mapped to UNMAPPED carry STATUS_UNMAPPED)

## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## Step 1 — Load lookup tables

In [ ]:
lookups = {
    'POS':         read_bronze('pos_status_codes'),         # status_code, description
    'EC':          read_bronze('ec_order_status_codes'),    # status_code, description
    'PG':          read_bronze('pg_status_codes'),          # status_code, description
    'MARKETPLACE': read_bronze('ts_seller_status_codes'),   # status_code, description
}

for sys_name, df in lookups.items():
    print(f'\n=== {sys_name} status codes ===')
    df.show(truncate=False)

## Step 2 — Define canonical mapping

Lead curates this. Each row maps `(source_system, raw_code) → canonical_status`. Inspect the lookup output above and fill in based on `docs/glossary.md` canonical set.

In [ ]:
# TODO: Lead — verify these mappings against the actual lookup outputs above.
# Add or correct rows; the mapping is the single source of truth for T2.
MAPPINGS = [
    # (source_system, raw_code, canonical_status, confidence)
    ('POS', 'OPN', 'OPEN',          1.00),
    ('POS', 'CMP', 'CONFIRMED',     1.00),
    ('POS', 'ABT', 'CANCELLED',     1.00),
    ('POS', 'RFD', 'REFUNDED',      1.00),
    ('POS', 'VOD', 'CANCELLED',     1.00),
    ('POS', 'HLD', 'OPEN',          1.00),

    ('EC', 'pending',    'OPEN',          1.00),
    ('EC', 'processing', 'OPEN',          1.00),
    ('EC', 'confirmed',  'CONFIRMED',     1.00),
    ('EC', 'picked',     'IN_FULFILMENT', 1.00),
    ('EC', 'packed',     'IN_FULFILMENT', 1.00),
    ('EC', 'shipped',    'IN_FULFILMENT', 1.00),
    ('EC', 'delivered',  'DELIVERED',     1.00),
    ('EC', 'cancelled',  'CANCELLED',     1.00),
    ('EC', 'returned',   'RETURNED',      1.00),

    ('PG', 'INITIATED',  'OPEN',          1.00),
    ('PG', 'AUTHORIZED', 'OPEN',          1.00),
    ('PG', 'CAPTURED',   'CONFIRMED',     1.00),
    ('PG', 'FAILED',     'CANCELLED',     1.00),
    ('PG', 'REVERSED',   'REFUNDED',      1.00),
    ('PG', 'DISPUTED',   'OPEN',          1.00),
    ('PG', 'PENDING',    'OPEN',          1.00),
    ('PG', 'REFUNDED',   'REFUNDED',      1.00),

    ('MARKETPLACE', 'ACTIVE',    'OPEN',      1.00),
    ('MARKETPLACE', 'SUSPENDED', 'CANCELLED', 1.00),
    ('MARKETPLACE', 'BANNED',    'CANCELLED', 1.00),
    ('MARKETPLACE', 'PENDING',   'OPEN',      1.00),
    ('MARKETPLACE', 'TERMINATED','CANCELLED', 1.00),
]

mapping_df = spark.createDataFrame(
    [Row(source_system=s, raw_code=c, canonical_status=cs, mapping_confidence=conf)
     for s, c, cs, conf in MAPPINGS]
)
mapping_df.show()

## Step 3 — Detect unmapped status codes

Compare distinct status values seen in transactional tables against the mapping. Anything not covered gets a row with `canonical_status='UNMAPPED'` and flagged for lead attention.

In [ ]:
# Distinct status values seen in transactional tables
transactional_sources = [
    ('POS',         'pos_transactions',  'status_code'),
    ('EC',          'ec_orders',          'order_status'),
    ('PG',          'pg_transactions',    'status_code'),
    ('MARKETPLACE', 'ts_sellers',         'seller_status'),
    # Add others as discovered
]

seen_rows = []
for sys_name, table, col in transactional_sources:
    distinct = read_bronze(table).select(F.col(col).alias('raw_code')).distinct().collect()
    for r in distinct:
        if r.raw_code is not None:
            seen_rows.append(Row(source_system=sys_name, raw_code=r.raw_code))

seen_df = spark.createDataFrame(seen_rows).distinct()

# Anti-join to find what's missing from MAPPINGS
unmapped = (seen_df
            .join(mapping_df, ['source_system', 'raw_code'], 'left_anti')
            .withColumn('canonical_status', F.lit('UNMAPPED'))
            .withColumn('mapping_confidence', F.lit(0.0)))

print(f'Found {unmapped.count()} unmapped status codes.')
unmapped.show()

## Step 4 — Combine + flag + write

Mapped rows = CLEAN. Unmapped rows = FLAGGED_ANOMALY with `STATUS_UNMAPPED`.

In [ ]:
combined = mapping_df.unionByName(unmapped, allowMissingColumns=True)
combined = add_anomaly_columns(combined)
combined = flag(
    combined,
    condition=(F.col('canonical_status') == 'UNMAPPED'),
    reason_code='STATUS_UNMAPPED',
    status='FLAGGED_ANOMALY',
    certainty='UNRELIABLE',
)

(combined.write.format('snowflake').options(**sf_silver)
    .option('dbtable', 'silver_status_code_mapping')
    .mode('overwrite').save())

print('silver_status_code_mapping written.')
combined.show(truncate=False)

## Lead checklist after run

1. Inspect any `STATUS_UNMAPPED` rows above — extend `MAPPINGS` and re-run if real codes are missing.
2. Post in team channel: "silver_status_code_mapping is live; T2 unblocked for all members."
3. Members import via:
   ```python
   mapping = (spark.read.format('snowflake').options(**sf_silver)
              .option('dbtable', 'silver_status_code_mapping').load())
   df_with_canonical = df.join(mapping,
       on=[df.source_system == mapping.source_system, df.raw_status == mapping.raw_code],
       how='left')
   ```